In [1]:
# ============================================================
# 1 — Imports & Configuration
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "Data" / "processed" / "project1_train.csv"
VAL_PATH   = PROJECT_ROOT / "Data" / "processed" / "project1_validation.csv"
TEST_PATH  = PROJECT_ROOT / "Data" / "processed" / "project1_test.csv"

train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VAL_PATH)
test = pd.read_csv(TEST_PATH)

TARGET = "Product"
TEXT_COL = "Consumer complaint narrative"

X_train = train[TEXT_COL].fillna("")
y_train = train[TARGET]

X_val = validation[TEXT_COL].fillna("")
y_val = validation[TARGET]

X_test = test[TEXT_COL].fillna("")
y_test = test[TARGET]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (58422,)
Validation: (11685,)
Test: (11685,)


In [2]:
# ============================================================
# 2 — Majority Class Baseline
# ============================================================

majority_class = y_train.mode()[0]

majority_predictions = np.full(
    len(y_val),
    majority_class
)

majority_accuracy = accuracy_score(
    y_val,
    majority_predictions
)

majority_macro_f1 = f1_score(
    y_val,
    majority_predictions,
    average="macro"
)

majority_weighted_f1 = f1_score(
    y_val,
    majority_predictions,
    average="weighted"
)

print("Majority class:", majority_class)
print(f"Accuracy:     {majority_accuracy:.4f}")
print(f"Macro F1:     {majority_macro_f1:.4f}")
print(f"Weighted F1:  {majority_weighted_f1:.4f}")

Majority class: Debt collection
Accuracy:     0.3293
Macro F1:     0.0450
Weighted F1:  0.1632


In [3]:
# ============================================================
# 3 — TF-IDF + Logistic Regression
# ============================================================

tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        )
    )
])

print("Training TF-IDF + Logistic Regression...")

tfidf_lr.fit(
    X_train,
    y_train
)

print("Training complete.")

Training TF-IDF + Logistic Regression...
Training complete.


### Validation predictions

In [4]:
# ============================================================
# 4 — Validation Evaluation
# ============================================================

val_predictions = tfidf_lr.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

val_macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro"
)

val_weighted_f1 = f1_score(
    y_val,
    val_predictions,
    average="weighted"
)

print(f"Accuracy:     {val_accuracy:.4f}")
print(f"Macro F1:     {val_macro_f1:.4f}")
print(f"Weighted F1:  {val_weighted_f1:.4f}")

Accuracy:     0.8340
Macro F1:     0.7144
Weighted F1:  0.8264


### Per-class performance

In [5]:
# ============================================================
# 5 — Classification Report
# ============================================================

print(
    classification_report(
        y_val,
        val_predictions,
        digits=4
    )
)

                                                         precision    recall  f1-score   support

                            Checking or savings account     0.7490    0.8740    0.8067      2096
                                            Credit card     0.8034    0.8421    0.8223      1970
    Credit reporting or other personal consumer reports     0.8509    0.6939    0.7644       477
                                        Debt collection     0.8789    0.9470    0.9117      3848
                              Debt or credit management     1.0000    0.2039    0.3387       103
     Money transfer, virtual currency, or money service     0.7797    0.6171    0.6890       935
                                               Mortgage     0.9378    0.9304    0.9341       762
Payday loan, title loan, personal loan, or advance loan     0.7296    0.4782    0.5777       412
                                           Prepaid card     0.9231    0.1846    0.3077       130
                             

In [6]:
# ============================================================
# 6 — Confusion Matrix
# ============================================================

labels = sorted(y_train.unique())

cm = confusion_matrix(
    y_val,
    val_predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

cm_df

,Checking or savings account,Credit card,Credit reporting or other personal consumer reports,Debt collection,Debt or credit management,"Money transfer, virtual currency, or money service",Mortgage,"Payday loan, title loan, personal loan, or advance loan",Prepaid card,Student loan,Vehicle loan or lease
Checking or savings account,1832,105,0,32,0,117,3,2,1,0,4
Credit card,138,1659,13,112,0,25,4,15,1,1,2
Credit reporting or other personal consumer reports,0,20,331,121,0,0,1,1,0,1,2
Debt collection,21,90,31,3644,0,2,9,16,0,5,30
Debt or credit management,10,18,3,39,21,1,3,5,0,1,2
"Money transfer, virtual currency, or money service",303,40,0,8,0,577,3,1,0,0,3
Mortgage,14,10,1,19,0,0,709,6,0,2,1
"Payday loan, title loan, personal loan, or advance loan",46,61,3,65,0,6,13,197,0,3,18
Prepaid card,56,38,0,2,0,9,0,1,24,0,0
Student loan,7,8,3,29,0,0,4,11,0,322,0


In [7]:
# ============================================================
# 7 — Normalized Confusion Matrix
# ============================================================

cm_normalized = (
    cm_df
    .div(cm_df.sum(axis=1), axis=0)
    * 100
)

cm_normalized.round(2)

,Checking or savings account,Credit card,Credit reporting or other personal consumer reports,Debt collection,Debt or credit management,"Money transfer, virtual currency, or money service",Mortgage,"Payday loan, title loan, personal loan, or advance loan",Prepaid card,Student loan,Vehicle loan or lease
Checking or savings account,87.40,5.01,0.00,1.53,0.00,5.58,0.14,0.10,0.05,0.00,0.19
Credit card,7.01,84.21,0.66,5.69,0.00,1.27,0.20,0.76,0.05,0.05,0.10
Credit reporting or other personal consumer reports,0.00,4.19,69.39,25.37,0.00,0.00,0.21,0.21,0.00,0.21,0.42
Debt collection,0.55,2.34,0.81,94.70,0.00,0.05,0.23,0.42,0.00,0.13,0.78
Debt or credit management,9.71,17.48,2.91,37.86,20.39,0.97,2.91,4.85,0.00,0.97,1.94
"Money transfer, virtual currency, or money service",32.41,4.28,0.00,0.86,0.00,61.71,0.32,0.11,0.00,0.00,0.32
Mortgage,1.84,1.31,0.13,2.49,0.00,0.00,93.04,0.79,0.00,0.26,0.13
"Payday loan, title loan, personal loan, or advance loan",11.17,14.81,0.73,15.78,0.00,1.46,3.16,47.82,0.00,0.73,4.37
Prepaid card,43.08,29.23,0.00,1.54,0.00,6.92,0.00,0.77,18.46,0.00,0.00
Student loan,1.82,2.08,0.78,7.55,0.00,0.00,1.04,2.86,0.00,83.85,0.00


In [8]:
# ============================================================
# 8 — Baseline Comparison
# ============================================================

baseline_results = pd.DataFrame([
    {
        "Model": "Majority Class",
        "Accuracy": majority_accuracy,
        "Macro F1": majority_macro_f1,
        "Weighted F1": majority_weighted_f1
    },
    {
        "Model": "TF-IDF + Logistic Regression",
        "Accuracy": val_accuracy,
        "Macro F1": val_macro_f1,
        "Weighted F1": val_weighted_f1
    }
])

baseline_results.round(4)

,Model,Accuracy,Macro F1,Weighted F1
0,Majority Class,0.3293,0.0450,0.1632
1,TF-IDF + Logistic Regression,0.8340,0.7144,0.8264


### Inspect strongest model features

- Look at what vocabulary the model actually learned.

In [9]:
# ============================================================
# 9 — Most Important Terms by Product
# ============================================================

vectorizer = tfidf_lr.named_steps["tfidf"]
classifier = tfidf_lr.named_steps["classifier"]

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

coefficients = classifier.coef_

for class_index, product in enumerate(classifier.classes_):

    top_indices = np.argsort(
        coefficients[class_index]
    )[-15:][::-1]

    top_terms = feature_names[top_indices]

    print("\n" + "=" * 80)
    print(product)
    print("=" * 80)
    print(", ".join(top_terms))


Checking or savings account
bank, chime, checking, funds, the bank, my account, account, deposit, checking account, chase, banking, overdraft, debit, deposited, debit card

Credit card
card, credit card, credit, synchrony, citi, capital one, interest, charge, citibank, barclays, billing, capital, american express, card account, charges

Credit reporting or other personal consumer reports
inquiries, accounts, my credit, transunion, credit report, inaccurate, report, equifax, late, these, experian, credit, reporting, late payment, xxxx balance

Debt collection
debt, collection, collections, the debt, owe, this debt, collector, validation, credit, collect, owed, bill, court, to collect, my credit

Debt or credit management
debt, program, debt relief, settlement, the program, fees, enrolled, debt settlement, relief, creditors, my creditors, blocked and, paid, beyond finance, consumer under

Money transfer, virtual currency, or money service
paypal, transfer, money, cashapp, transaction, a

### Error inspection

In [10]:
# ============================================================
# 10 — Validation Errors
# ============================================================

validation_results = validation[
    [TARGET, TEXT_COL]
].copy()

validation_results["predicted_product"] = val_predictions

errors = validation_results[
    validation_results[TARGET]
    != validation_results["predicted_product"]
].copy()

print("Validation rows:", len(validation_results))
print("Errors:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(validation_results) * 100, 2),
    "%"
)

errors.head(20)

Validation rows: 11685
Errors: 1940
Error rate: 16.6 %


,Product,Consumer complaint narrative,predicted_product
1,Credit card,"On XX/XX/year>, Chime financial greyed out my ...",Checking or savings account
18,Vehicle loan or lease,"I am owed a refund for an extended warranty, i...",Checking or savings account
30,Vehicle loan or lease,I received credit alerts indicating that JD By...,Debt collection
50,Checking or savings account,In XXXX of XXXX I received a copy of my consum...,Debt collection
66,"Money transfer, virtual currency, or money ser...",Navy Federal is violating regulation E by deny...,Checking or savings account
77,"Money transfer, virtual currency, or money ser...",Fraudulent withdrawals from my account # XXXX ...,Checking or savings account
90,Checking or savings account,"On XX/XX/XXXX, my business was the victim of a...","Money transfer, virtual currency, or money ser..."
97,Prepaid card,"On XX/XX/year>, I purchased a XXXX XXXX XXXX f...",Credit card
99,"Money transfer, virtual currency, or money ser...","On XX/XX/XXXX, I created a Dave account solely...",Checking or savings account
115,Credit reporting or other personal consumer re...,I expect a thorough investigation into these m...,Debt collection


In [11]:
# ============================================================
# 11 — Most Common Confusion Pairs
# ============================================================

confusion_pairs = (
    errors
    .groupby([TARGET, "predicted_product"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs.head(20)

,Product,predicted_product,count
39,"Money transfer, virtual currency, or money ser...",Checking or savings account,303
7,Credit card,Checking or savings account,138
17,Credit reporting or other personal consumer re...,Debt collection,121
2,Checking or savings account,"Money transfer, virtual currency, or money ser...",117
9,Credit card,Debt collection,112
0,Checking or savings account,Credit card,105
23,Debt collection,Credit card,90
74,Vehicle loan or lease,Debt collection,75
55,"Payday loan, title loan, personal loan, or adv...",Debt collection,65
53,"Payday loan, title loan, personal loan, or adv...",Credit card,61



### Baseline Analysis Summary :
### PROJECT 1 — BASELINE MODEL
===========================

Target:
    Product

Prediction point:
    Date received

Feature representation:
    TF-IDF word unigrams + bigrams

Model:
    Logistic Regression

Baseline:
    Majority Class

Primary metric:
    Macro F1

Secondary metrics:
    Weighted F1
    Accuracy
    Per-class Precision / Recall
    Confusion Matrix

Next investigation:
- 1. Is TF-IDF sufficiently strong?
    - 2. Which Product classes are confused?
    - 3. Are errors caused by vocabulary overlap?
    - 4. Does character n-gram TF-IDF improve robustness?
    - 5. Does legitimate metadata improve performance?
